# Table 1 — cohort characteristics

Builds Table 1 of the manuscript ("Quantum or just quantised?") from files already in Drive.
**CPU runtime is fine. Runtime about 5 minutes.**

What it does
1. Finds the 1,200 Nellore series that were actually analysed (the rows of the B1 feature file)
   and joins them to the cohort table that carries age, sex, scanner, slice thickness and date.
2. Finds the 3,889 RSNA series that were analysed and summarises what the public release contains.
3. Checks itself against numbers already reported in the paper (1,200 series, 1,178 patients,
   3,889 series, 94 / 859 / 1,598 positives) and says PASS or MISMATCH for each.
4. Writes `table1_cohorts.csv` to `rsna_q3d_output/figures_paper/`.

**Privacy.** Nothing in this notebook prints an identifier, a name, a date of birth or a single
patient's value. Every output is a count, a median or a range, so the saved outputs are safe to
keep and to share.

In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# 1. Mount Drive and set paths ----------------------------------------------------------------
import os, re, json, warnings
warnings.filterwarnings("ignore")
from qct3d.config import mount_drive_if_colab
mount_drive_if_colab(P)

RSNA_DIR  = P["rsna_output_dir"]
INDIA_DIR = P["external_output_dir"]
OUT_DIR   = os.path.join(RSNA_DIR, "figures_paper")
# The Nellore cohort table. Both copies were written from the same CSV; the first that exists wins.
MATCHED_CANDIDATES = [os.path.join(P["pillar0_external_output_dir_v1"], "matched_table.parquet"),
                      os.path.join(P["pillar0_external_output_dir"], "matched_table.parquet")]
BACKBONE_FILE = "features_B1_swinunetr_suprem.parquet"     # defines the analysed series

# numbers already reported in the manuscript -- used only as self-checks
EXPECT = {"india_series": 1200, "india_patients": 1178, "india_hem": 94, "india_any": 859,
          "rsna_series": 3889, "rsna_hem": 1598}
os.makedirs(OUT_DIR, exist_ok=True)
for p in [RSNA_DIR, INDIA_DIR] + MATCHED_CANDIDATES:
    print(("found   " if os.path.exists(p) else "MISSING ") + p)


In [ ]:
# 2. Helpers -- schema-only inspection, joins, robust parsing -----------------------------------
import numpy as np, pandas as pd, pyarrow.parquet as pq

FEATURE_NAME = re.compile(r"^(f|x|feat|feature|emb|e|z|v)_?\d+$", re.I)

def id_columns(path):
    """Columns that are not feature vectors: scalar, and not named f0/f1/... ."""
    sch = pq.read_schema(path)
    keep = []
    for f in sch:
        t = str(f.type)
        if FEATURE_NAME.match(f.name) or t.startswith(("list", "large_list", "fixed_size_list")):
            continue
        keep.append(f.name)
    return keep

def describe(path, label):
    """Print shape and column names/types only -- never values."""
    pf = pq.ParquetFile(path)
    cols = id_columns(path)
    print(f"\n{label}: {pf.metadata.num_rows:,} rows, {pf.metadata.num_columns:,} columns "
          f"({len(cols)} non-feature)")
    sch = pq.read_schema(path)
    print("  ", ", ".join(f"{c}:{sch.field(c).type}" for c in cols[:60]),
          "..." if len(cols) > 60 else "")
    return cols

def read_cols(path, cols):
    return pq.read_table(path, columns=list(cols)).to_pandas()

UID = re.compile(r"\d+(?:\.\d+){4,}")
def norm_keys(s):
    """Normalise a key column: string, stripped; for paths also keep the basename."""
    s = s.astype(str).str.strip()
    return s

def key_forms(v):
    v = str(v).strip()
    out = {v}
    if "/" in v or "\\" in v:
        b = re.split(r"[\\/]", v.rstrip("/\\"))[-1]
        out.add(b)
    out.update(UID.findall(v))
    return out

def best_join(left, right, lcands, rcands, need):
    """Pick the (left col, right col) pair whose key forms overlap most. Returns (lc, rc, map)."""
    best = (None, None, {}, 0)
    for lc in [c for c in lcands if c in left.columns]:
        lvals = left[lc].astype(str)
        for rc in [c for c in rcands if c in right.columns]:
            index = {}
            for i, v in right[rc].astype(str).items():
                for k in key_forms(v):
                    index.setdefault(k, i)
            m = {}
            for i, v in lvals.items():
                for k in key_forms(v):
                    if k in index:
                        m[i] = index[k]; break
            if len(m) > best[3]:
                best = (lc, rc, m, len(m))
    lc, rc, m, n = best
    print(f"  join: features.{lc}  <->  table.{rc}   matched {n:,} of {len(left):,}")
    if n < need:
        raise RuntimeError(f"Only {n} of {len(left)} rows could be joined; expected {need}. "
                           "Check which matched_table / feature file nb5 used.")
    return lc, rc, m

def first_present(df, names):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

AGE = re.compile(r"(\d+(?:\.\d+)?)\s*([YMWD])?", re.I)
def age_years(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    m = AGE.search(str(v))
    if not m:
        return np.nan
    x, u = float(m.group(1)), (m.group(2) or "Y").upper()
    x = x / {"Y": 1, "M": 12, "W": 52.18, "D": 365.25}[u]
    return x if 0 <= x <= 110 else np.nan

def sex_code(v):
    s = str(v).strip().upper()[:1]
    return s if s in ("M", "F") else np.nan

def med_iqr(x, fmt="{:.0f}"):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    if len(x) == 0:
        return "not available"
    q1, q2, q3 = np.percentile(x, [25, 50, 75])
    return f"{fmt.format(q2)} ({fmt.format(q1)}–{fmt.format(q3)})"

def n_pct(k, n):
    return f"{k:,} ({100.0 * k / n:.1f})"

def check(name, got, want):
    ok = (got == want)
    print(f"  {'PASS    ' if ok else 'MISMATCH'} {name}: {got} (manuscript: {want})")
    return ok

print("helpers ready")


In [ ]:
# 3. Nellore (external) cohort --------------------------------------------------------------------
india_feat_path = os.path.join(INDIA_DIR, BACKBONE_FILE)
matched_path = next((p for p in MATCHED_CANDIDATES if os.path.exists(p)), None)
assert os.path.exists(india_feat_path), india_feat_path
assert matched_path, "no matched_table.parquet found"

fcols = describe(india_feat_path, "Nellore analysed series (B1 features)")
mcols = describe(matched_path, "Nellore cohort table")
ifeat = read_cols(india_feat_path, fcols)

WANT = ["SeriesInstanceUID", "StudyInstanceUID", "PatientID", "PatientAge", "MatchedReportAge",
        "PatientSex", "MatchedReportSex", "SliceThickness", "Manufacturer",
        "ManufacturerModelName", "KVP", "StudyDate", "MatchedReportDate", "NumSlices",
        "Rows", "Columns", "SourceFolder", "DiskPath", "FullFolderPath", "AccessionNumber"]
mtab = read_cols(matched_path, [c for c in WANT if c in mcols])

lc, rc, m = best_join(
    ifeat, mtab,
    lcands=[c for c in ["series", "series_uid", "SeriesInstanceUID", "series_id", "uid", "path",
                        "series_path", "folder", "study"] if c in ifeat.columns] + list(ifeat.columns),
    rcands=["SeriesInstanceUID", "SourceFolder", "FullFolderPath", "DiskPath", "StudyInstanceUID",
            "AccessionNumber"],
    need=int(0.98 * len(ifeat)))
ind = mtab.loc[[m[i] for i in ifeat.index if i in m]].reset_index(drop=True)

# labels, if the feature file carries them
def find_label(df, names):
    c = first_present(df, names)
    if c is None: return None
    v = pd.to_numeric(df[c], errors="coerce")
    return int((v > 0.5).sum()) if v.notna().any() else None
i_hem = find_label(ifeat, ["hemorrhage", "y_hemorrhage", "label_hemorrhage", "ich", "y_ich"])
i_any = find_label(ifeat, ["abnormal_any", "any_finding", "y_abnormal_any", "label_abnormal_any"])

n_series = len(ifeat)
pid = first_present(ind, ["PatientID"])
n_pat = int(ind[pid].nunique()) if pid else None
age = ind[first_present(ind, ["PatientAge"])].map(age_years) if first_present(ind, ["PatientAge"]) else pd.Series(np.nan, index=ind.index)
if first_present(ind, ["MatchedReportAge"]):
    age = age.fillna(ind[first_present(ind, ["MatchedReportAge"])].map(age_years))
sex = ind[first_present(ind, ["PatientSex"])].map(sex_code) if first_present(ind, ["PatientSex"]) else pd.Series(np.nan, index=ind.index)
if first_present(ind, ["MatchedReportSex"]):
    sex = sex.fillna(ind[first_present(ind, ["MatchedReportSex"])].map(sex_code))
# one value per patient for age and sex, so repeat studies do not double-count a person
if pid:
    per_pat = pd.DataFrame({"p": ind[pid].astype(str), "age": age, "sex": sex}).groupby("p").first()
    age_p, sex_p = per_pat["age"], per_pat["sex"]
else:
    age_p, sex_p = age, sex
thick = pd.to_numeric(ind.get("SliceThickness"), errors="coerce") if "SliceThickness" in ind else pd.Series(dtype=float)
nsl = pd.to_numeric(ind.get("NumSlices"), errors="coerce") if "NumSlices" in ind else pd.Series(dtype=float)
kvp = pd.to_numeric(ind.get("KVP"), errors="coerce") if "KVP" in ind else pd.Series(dtype=float)
scanner = "not available"
if "Manufacturer" in ind:
    s = (ind["Manufacturer"].astype(str).str.strip() + " " +
         (ind["ManufacturerModelName"].astype(str).str.strip() if "ManufacturerModelName" in ind else "")).str.strip()
    vc = s[s.str.lower().ne("nan") & s.ne("")].value_counts()
    scanner = "; ".join(f"{k} ({100 * v / vc.sum():.0f}%)" for k, v in vc.head(3).items()) if len(vc) else "not available"
dcol = first_present(ind, ["StudyDate", "MatchedReportDate"])
period, months = "not available", None
if dcol:
    d = pd.to_datetime(ind[dcol].astype(str).str.slice(0, 8), format="%Y%m%d", errors="coerce").dropna()
    if len(d):
        period = f"{d.min():%b %Y} – {d.max():%b %Y}"
        months = d.dt.to_period("M").value_counts().sort_index()

INDIA = {
    "series": f"{n_series:,}",
    "patients": f"{n_pat:,}" if n_pat else "not available",
    "age": med_iqr(age_p),
    "age_missing": f"{int(age_p.isna().sum())} of {len(age_p)}",
    "female": n_pct(int((sex_p == "F").sum()), int(sex_p.notna().sum())) if sex_p.notna().any() else "not available",
    "sex_missing": f"{int(sex_p.isna().sum())} of {len(sex_p)}",
    "scanner": scanner,
    "kvp": med_iqr(kvp),
    "thickness": med_iqr(thick, "{:.1f}"),
    "slices": med_iqr(nsl),
    "period": period,
    "hem": n_pct(i_hem if i_hem is not None else EXPECT["india_hem"], n_series),
    "any": n_pct(i_any if i_any is not None else EXPECT["india_any"], n_series),
}
print("\nNellore summary (aggregates only):")
for k, v in INDIA.items():
    print(f"  {k:<12} {v}")
if months is not None:
    print("  studies per calendar month:", ", ".join(f"{p}: {c}" for p, c in months.items()))
print("\nself-checks against the manuscript")
check("Nellore series", n_series, EXPECT["india_series"])
if n_pat: check("Nellore patients", n_pat, EXPECT["india_patients"])
if i_hem is not None: check("Nellore hemorrhage positives", i_hem, EXPECT["india_hem"])
else: print("  (no hemorrhage label in the feature file -- manuscript value used)")
if i_any is not None: check("Nellore any-finding positives", i_any, EXPECT["india_any"])
else: print("  (no any-finding label in the feature file -- manuscript value used)")


In [ ]:
# 4. RSNA (development) cohort --------------------------------------------------------------------
rsna_feat_path = os.path.join(RSNA_DIR, BACKBONE_FILE)
series_path = os.path.join(RSNA_DIR, "series_table.parquet")
header_path = os.path.join(RSNA_DIR, "header_index.parquet")
assert os.path.exists(rsna_feat_path), rsna_feat_path

rfcols = describe(rsna_feat_path, "RSNA analysed series (B1 features)")
rfeat = read_cols(rsna_feat_path, rfcols)
# Phase A admitted 68 series from an earlier pilot cohort into the B1 file. They carry no
# label and no patient id, and every analysis excluded them (nb2/nb6 use the intersection
# of the B1-B3 tables, 3,889 series). Drop them here as well.
_lab = first_present(rfeat, ["any"])
if _lab is not None:
    _n0 = len(rfeat)
    rfeat = rfeat[pd.to_numeric(rfeat[_lab], errors="coerce").notna()].reset_index(drop=True)
    print(f"  dropped {_n0 - len(rfeat)} unlabelled out-of-cohort series (expected 68)")
st = None
if os.path.exists(series_path):
    scols = describe(series_path, "RSNA series table")
    st = read_cols(series_path, scols)
hdr_cols = describe(header_path, "RSNA per-slice header index") if os.path.exists(header_path) else []

r_key_cands = [c for c in ["series", "series_uid", "SeriesInstanceUID", "series_id", "uid"] if c in rfeat.columns] + list(rfeat.columns)
rs = None
if st is not None:
    lc2, rc2, m2 = best_join(rfeat, st, r_key_cands,
                             ["SeriesInstanceUID", "series", "series_uid", "series_id", "uid"],
                             need=int(0.98 * len(rfeat)))
    rs = st.loc[[m2[i] for i in rfeat.index if i in m2]].reset_index(drop=True)
else:
    rs = rfeat.copy()

r_n = len(rfeat)
r_hem = find_label(rfeat, ["any", "y_any", "label_any", "label", "y", "target", "hemorrhage", "ich"])
if r_hem is None and rs is not None:
    r_hem = find_label(rs, ["any", "y_any", "label_any", "label", "y", "target", "hemorrhage", "ich"])

def from_either(names):
    for df in (rs, rfeat):
        if df is not None:
            c = first_present(df, names)
            if c: return df[c]
    return None

r_pid = from_either(["PatientID", "patient", "patient_id"])
r_nsl = from_either(["n_slices", "num_slices", "NumSlices", "slices", "n"])
r_thk = from_either(["SliceThickness", "slice_thickness", "dz", "slice_spacing", "z_spacing"])
r_ps = from_either(["pixel_spacing", "PixelSpacing", "dx", "spacing_xy", "inplane"])
r_age = from_either(["PatientAge", "age"])
r_sex = from_either(["PatientSex", "sex"])
r_man = from_either(["Manufacturer", "manufacturer"])

# fall back to the per-slice header index for slice counts, spacing and patient ids
if os.path.exists(header_path) and (r_nsl is None or r_thk is None or r_pid is None):
    want = [c for c in hdr_cols if c in ("SeriesInstanceUID", "series", "series_uid", "PatientID",
                                         "ImagePositionPatient", "z", "pos_z", "ipp_z",
                                         "PixelSpacing", "PatientAge", "PatientSex", "Manufacturer")]
    sch = pq.read_schema(header_path)
    ipp_list = [c for c in sch.names if c in ("ImagePositionPatient",) and str(sch.field(c).type).startswith(("list", "large_list", "fixed_size_list"))]
    hdr = pq.read_table(header_path, columns=want + ipp_list).to_pandas()
    skey = first_present(hdr, ["SeriesInstanceUID", "series", "series_uid"])
    keep = set()
    for v in rfeat[[c for c in r_key_cands if c in rfeat.columns][0]].astype(str):
        keep |= key_forms(v)
    hdr = hdr[hdr[skey].astype(str).isin(keep)]
    print(f"  header rows for analysed series: {len(hdr):,} across {hdr[skey].nunique():,} series")
    g = hdr.groupby(skey)
    if r_nsl is None:
        r_nsl = g.size()
    zc = first_present(hdr, ["z", "pos_z", "ipp_z"])
    if zc is None and ipp_list:
        hdr["_z"] = hdr["ImagePositionPatient"].map(lambda a: float(a[2]) if a is not None and len(a) > 2 else np.nan)
        zc = "_z"
    if r_thk is None and zc:
        r_thk = g[zc].apply(lambda z: np.median(np.diff(np.sort(pd.to_numeric(z, errors="coerce").dropna()))) if len(z) > 1 else np.nan)
    if r_pid is None and first_present(hdr, ["PatientID"]):
        r_pid = g[first_present(hdr, ["PatientID"])].first()
    if r_age is None and first_present(hdr, ["PatientAge"]):
        r_age = g[first_present(hdr, ["PatientAge"])].first()
    if r_sex is None and first_present(hdr, ["PatientSex"]):
        r_sex = g[first_present(hdr, ["PatientSex"])].first()
    if r_man is None and first_present(hdr, ["Manufacturer"]):
        r_man = g[first_present(hdr, ["Manufacturer"])].first()

def scan_summary(s):
    if s is None: return "not recorded in the public release"
    s = s.astype(str).str.strip()
    vc = s[s.str.lower().ne("nan") & s.ne("")].value_counts()
    return "; ".join(f"{k} ({100 * v / vc.sum():.0f}%)" for k, v in vc.head(3).items()) if len(vc) else "not recorded in the public release"

def coverage(s):
    """Fraction of analysed RSNA series for which a per-series value exists."""
    if s is None: return 0.0
    return float(pd.Series(s).notna().sum()) / r_n

def guarded(s, render, what):
    """Render a statistic only if it covers the analysed series; otherwise say how much it covers."""
    if s is None:
        return "not available"
    cov = coverage(s)
    if cov >= 0.98:
        return render(s)
    print(f"  WARNING {what}: available for only {int(round(cov * r_n)):,} of {r_n:,} series -- reported as partial")
    return f"{render(s)} [partial: {int(round(cov * r_n)):,} of {r_n:,} series]"

thk_from_positions = (r_thk is not None and getattr(r_thk, "name", "") not in
                      ("SliceThickness", "slice_thickness"))
r_age_y = r_age.map(age_years) if r_age is not None else None
r_sex_c = r_sex.map(sex_code) if r_sex is not None else None
RSNA = {
    "series": f"{r_n:,}",
    "patients": (f"{int(pd.Series(r_pid).nunique()):,}" if r_pid is not None and coverage(r_pid) >= 0.98
                 else guarded(r_pid, lambda x: f"{int(pd.Series(x).nunique()):,}", "RSNA patient ids")),
    "age": med_iqr(r_age_y) if r_age_y is not None and r_age_y.notna().any() else "not released",
    "female": (n_pct(int((r_sex_c == "F").sum()), int(r_sex_c.notna().sum()))
               if r_sex_c is not None and r_sex_c.notna().any() else "not released"),
    "scanner": scan_summary(r_man) if r_man is not None else "multi-institutional; not recorded in the public release",
    "kvp": "not recorded in the public release",
    "thickness": guarded(r_thk, lambda x: med_iqr(x, "{:.1f}") + (" (from slice spacing)" if thk_from_positions else ""),
                         "RSNA slice thickness"),
    "slices": guarded(r_nsl, med_iqr, "RSNA slices per series"),
    "period": "not released",
    "hem": n_pct(r_hem if r_hem is not None else EXPECT["rsna_hem"], r_n),
    "any": "not defined",
}
print("\nRSNA summary (aggregates only):")
for k, v in RSNA.items():
    print(f"  {k:<12} {v}")
print("\nself-checks against the manuscript")
check("RSNA series", r_n, EXPECT["rsna_series"])
if r_hem is not None: check("RSNA hemorrhage positives", r_hem, EXPECT["rsna_hem"])
else: print("  (no hemorrhage label found -- manuscript value used)")


In [ ]:
# 5. Assemble Table 1 and save it -----------------------------------------------------------------
ROWS = [
    ("Source", "Multi-institutional public challenge dataset", "Routine clinical studies, one government hospital, India"),
    ("Series analysed, n", RSNA["series"], INDIA["series"]),
    ("Patients, n", RSNA["patients"], INDIA["patients"]),
    ("Age, years, median (IQR)", RSNA["age"], INDIA["age"]),
    ("Female sex, n (%)", RSNA["female"], INDIA["female"]),
    ("Scanner", RSNA["scanner"], INDIA["scanner"]),
    ("Tube voltage, kVp, median (IQR)", RSNA["kvp"], INDIA["kvp"]),
    ("Slice thickness, mm, median (IQR)", RSNA["thickness"], INDIA["thickness"]),
    ("Slices per series, median (IQR)", RSNA["slices"], INDIA["slices"]),
    ("Acquisition period", RSNA["period"], INDIA["period"]),
    ("Hemorrhage, n (%)", RSNA["hem"], INDIA["hem"]),
    ("Any reported finding, n (%)", RSNA["any"], INDIA["any"]),
    ("Reference standard", "Expert slice-level annotation", "Structured radiology-report flags"),
]
T1 = pd.DataFrame(ROWS, columns=["Characteristic", "Development cohort (RSNA-ICH)", "External cohort (Nellore)"])
out = os.path.join(OUT_DIR, "table1_cohorts.csv")
T1.to_csv(out, index=False)
notes = {"india_age_missing": INDIA["age_missing"], "india_sex_missing": INDIA["sex_missing"],
         "matched_table": os.path.relpath(matched_path, ROOT), "join_key": f"{lc} <-> {rc}"}
with open(os.path.join(OUT_DIR, "table1_cohorts_notes.json"), "w") as fh:
    json.dump(notes, fh, indent=1)
pd.set_option("display.max_colwidth", 80)
print(T1.to_string(index=False))
print(f"\nwritten: {out}")
print("Age and sex are counted once per patient; missing values are excluded from the median and the percentage.")
print("notes:", notes)
